# Analytics Engineering: practice test
## Weekly operations pipeline. **MODEL ANSWER**

Note - all comments from me are *in italics* and not part of the answer I am replicating here.

*This is one good answer, not the only one. In fact this is too good an answer, yo uare not expected to do exactly this by any stretch and there are lots of high scoring submissions that (a) do a LOT less than this and (b) don't ever get to creating silver tables, etc. Several choices below (time zone, what counts as an order, how to treat outliers) are decisions, and a different stated decision with matching checks is equally acceptable. It always depends.*

## The data

You have six tables from a fictional food-delivery marketplace, Plateful, which works with restaurants in five UK cities. They cover orders placed from 1 March to 30 June 2026.

| Name in this notebook | What it holds |
| --- | --- |
| `orders` | One record per order: customer, restaurant, when it was placed, delivery fee |
| `order_status_events` | The order's progress through the process, recorded as events |
| `order_lines` | The items inside each order |
| `payments` | Money movements for each order: charges and refunds |
| `restaurants` | Restaurant details: cuisine and city |
| `customers` | Customer details |

All timestamps in all tables are stored in **UTC**.

**How the tables connect**

- `orders` to `order_status_events`, `order_lines` and `payments` on `order_id`
- `orders` to `restaurants` on `restaurant_id`
- `orders` to `customers` on `customer_id`

Check the actual column names in the next cell before you write any joins.

In [1]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/practicesession/plateful-practice.zip"    # link to plateful-practice.zip (the CSV files are inside a folder in the zip)

DATA_DIR = "/content/plateful"

if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/plateful.zip")
    zipfile.ZipFile("/content/plateful.zip").extractall(DATA_DIR)

FILES = {
    "orders": "orders.csv",
    "order_status_events": "order_status_events.csv",
    "order_lines": "order_lines.csv",
    "payments": "payments.csv",
    "restaurants": "restaurants.csv",
    "customers": "customers.csv",
}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}.")

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))

orders: 14,494 rows, 5 columns
    ['order_id', 'customer_id', 'restaurant_id', 'order_ts', 'delivery_fee']
order_status_events: 56,812 rows, 4 columns
    ['event_id', 'order_id', 'status', 'event_ts']
order_lines: 36,120 rows, 5 columns
    ['order_id', 'line_no', 'item_name', 'quantity', 'unit_price']
payments: 16,423 rows, 5 columns
    ['payment_id', 'order_id', 'payment_type', 'amount', 'paid_at']
restaurants: 67 rows, 5 columns
    ['restaurant_id', 'restaurant_name', 'cuisine', 'city', 'effective_from']
customers: 4,200 rows, 3 columns
    ['customer_id', 'signup_date', 'home_city']


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each, using only the tables and keys described above. Run them and change them to see what they do. You do not have to use them in your answer.

In [2]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

# Look at the first few rows of a table
display(q("SELECT * FROM orders LIMIT 5"))

# How many rows are there, and how many different order_id values?
display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT order_id) AS n_distinct_orders FROM orders"))

# Join two tables on a key
display(q("""
    SELECT o.order_id, r.restaurant_id
    FROM orders AS o
    JOIN restaurants AS r ON o.restaurant_id = r.restaurant_id
    LIMIT 5
"""))

,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00


,n_rows,n_distinct_orders
0,14494,14494


,order_id,restaurant_id
0,O000001,R052
1,O000002,R027
2,O000003,R044
3,O000004,R049
4,O000005,R009


In [3]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

orders = tables["orders"]
display(orders.head())        # first few rows
print(orders.shape)           # (number of rows, number of columns)
print(orders.dtypes)          # data type of each column

# The same join as in Example 1, using pandas
joined = tables["orders"].merge(tables["restaurants"], on="restaurant_id", how="inner")
display(joined[["order_id", "restaurant_id"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT customer_id FROM orders")
print(result["customer_id"].nunique(), "different customer_id values in orders")

,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00


(14494, 5)
order_id          object
customer_id       object
restaurant_id     object
order_ts          object
delivery_fee     float64
dtype: object


,order_id,restaurant_id
0,O000001,R052
1,O000002,R027
2,O000003,R044
3,O000004,R049
4,O000005,R009


4073 different customer_id values in orders


## Your brief

**From:** Operations

> We run Plateful, a food-delivery marketplace in five UK cities. We are starting a weekly operations report and need the first layers of a data pipeline behind it: clean tables first, then one reporting table. For each week, show orders, net revenue and average delivery time by cuisine and by city. We also need to know how far we can trust the figures.

**What your final table should look like**

One table with columns for week, cuisine, city, number of orders, net revenue and average delivery time in minutes. State clearly what one row of your table represents.

**Decisions you must make and state**

- Which week does an order belong to?
- Which orders count as orders, and which do not?
- What does "net revenue" mean here, and where does it come from?
- What counts as "delivery time", and for which orders can it be measured?

**A suggested route, in order** (you are not expected to finish)

1. Profile the tables: grain/rows, keys, duplicates, odd values.
2. Build clean tables (a silver layer) with one row per order where that is the grain.
3. Build the one reporting table (gold) from them.
4. Show with checks that the gold table agrees with the source data.
5. If you have time: add cancelled orders and a refund rate to the table, and write automated checks for your clean tables.

## 1. Understand the data

*Look before you build. Write down what you find: what one row of each table represents, how the tables connect, and anything that looks odd or missing.*

*Prompts and checks can be written at the end (sections four and five respectively) or as you go along. I'll give examples of both approaches here.*

Strategy: for each table, find the grain (what one row is) and whether the key is unique, before any join.

Example prompt: "for each table in tables, loop through and print the table name, the number of rows in the table and the first three records."
I would expect this to be displayed as detailed, and that the tables look like described above and have a significant number of rows:

In [4]:
from IPython.display import display
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

for name in tables:
    t = tables[name]
    print(f"\n=== Table {name}: {t.shape[0]:,} rows ===")
    display(t.head(3))


=== Table orders: 14,494 rows ===


,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00



=== Table order_status_events: 56,812 rows ===


,event_id,order_id,status,event_ts
0,E027809,O007104,accepted,2026-04-29T13:28:57Z
1,E018209,O004658,placed,2026-04-08T19:11:30Z
2,E043792,O011178,accepted,2026-06-02T17:07:47Z



=== Table order_lines: 36,120 rows ===


,order_id,line_no,item_name,quantity,unit_price
0,O000001,1,Ramen,3,12.0
1,O000002,1,Soft drink,1,2.0
2,O000002,2,Side salad,1,3.5


... (2 outputs left out)


=== Table restaurants: 67 rows ===


,restaurant_id,restaurant_name,cuisine,city,effective_from
0,R064,Fresh Vegan 64,Vegan,London,2025-01-01
1,R012,Royal Burgers 12,Burgers,Bristol,2025-01-01
2,R028,Grand Burgers 28,Burgers,Birmingham,2025-01-01



=== Table customers: 4,200 rows ===


,customer_id,signup_date,home_city
0,C00001,2024-10-15,London
1,C00002,2025-05-17,Leeds
2,C00003,2026-01-25,Leeds


This seems to make sense in terms of the description. Tables look coherent and logical. "restaurants" is the only one with a limited number of rows (67) but that makes sense. Looks like standard relational database tables.

Now I'll check for missing values.
Prompt: "please loop through every table in tables and check if there are missing values - printing this information to the screen." I would expect this to give me the same number of outputs as the the last loop, and that most tables have most values included.

In [5]:
for name in tables:
    t = tables[name]
    print(f"Table {name}")
    print("missing values:", t.isna().sum()[t.isna().sum() > 0].to_dict() or "none")

Table orders
missing values: none
Table order_status_events
missing values: none
Table order_lines
missing values: none
Table payments
missing values: none
Table restaurants
missing values: none
Table customers
missing values: none


All tables listed and no missing values.

I can see in the tables above that each has an "id" or some kind which could logically be the grain/unique element of each row. Let's check if the are unique to each table.

Prompt: show me for each table the number of rows and the number of unique ids. The unique ids should be order_id for orders; payment_id for payments; event_id for order_status_events; order_id for order_status_events; restaurant_id for restaurants; customer_id for customers.

*You potentially don't need to specify all the id fields for the AI to get it right - just make sure you check they are all there. The order_status_events probably is event_id but could be order_id. I checked both.*

My expectation is a list of tables and the unique number of rows and unique id records - these should be the same number.

In [6]:
# Grain and key uniqueness: is the first column (or key pair) really unique?
display(q("""
SELECT 'orders.order_id' AS key, COUNT(*) AS n_rows, COUNT(DISTINCT order_id) AS n_distinct FROM orders
UNION ALL SELECT 'payments.payment_id', COUNT(*), COUNT(DISTINCT payment_id) FROM payments
UNION ALL SELECT 'order_status_events.event_id', COUNT(*), COUNT(DISTINCT event_id) FROM order_status_events
UNION ALL SELECT 'order_status_events.order_id', COUNT(*), COUNT(DISTINCT order_id) FROM order_status_events
UNION ALL SELECT 'restaurants.restaurant_id', COUNT(*), COUNT(DISTINCT restaurant_id) FROM restaurants
UNION ALL SELECT 'customers.customer_id', COUNT(*), COUNT(DISTINCT customer_id) FROM customers
"""))

,key,n_rows,n_distinct
0,orders.order_id,14494,14494
1,payments.payment_id,16423,16393
2,order_status_events.event_id,56812,56812
3,order_status_events.order_id,56812,14494
4,restaurants.restaurant_id,67,64
5,customers.customer_id,4200,4200


Findings so far. `orders` has one row per order. Three tables break the "one row per thing" assumption, and each would mess up the report if I joined naively:

- `payments`: `payment_id` is not unique, so some payment rows appear more than once.
- `order_status_events`: several rows per order by design (it is an event log), so the grain is *order and status*, not order. It also contains repeated rows.
- `restaurants`: `restaurant_id` is not unique, so a restaurant can have more than one row.

*We should go and look at each of these three individually, but I'll show you checking just a couple of them. You could check all these issues or you could identify the problem and list it as a task you need to look into further (i.e. skip it for now with the understanding that in the real world you would come back to investigate and fix.)*

Prompt: "find the items in restaurants that share the same restaurant_id. I.e. if restaurant_id = R001 is in the table twice, print both these rows."

I'm expecting to see up to three duplicated restaurant ids as the earlier output said there were 67 rows and 64 unique restaurant ids.

In [7]:
# Look at the odd cases directly rather than trusting the counts.
print("Restaurants with more than one row:")
display(q("SELECT * FROM restaurants WHERE restaurant_id IN (SELECT restaurant_id FROM restaurants GROUP BY 1 HAVING COUNT(*) > 1) ORDER BY restaurant_id, effective_from"))

Restaurants with more than one row:


,restaurant_id,restaurant_name,cuisine,city,effective_from
0,R007,Royal Japanese 7,Japanese,Bristol,2025-01-01
1,R007,Royal Japanese 7,Japanese,London,2026-04-14
2,R021,Grand Thai 21,Thai,Manchester,2025-01-01
3,R021,Grand Thai 21,Thai,LEEDS,2026-05-06
4,R040,Corner Vegan 40,Vegan,London,2025-01-01
5,R040,Corner Vegan 40,Vegan,Bristol,2026-05-27


So it looks like the duplications are for chains that have restaurants in more than one city. This means they unique element of a row is the restaurant_id and city combined.

I'll now look into the statuses and number of events in the order_status_events table.

*This example has a prompt asking for two things at the same time. This is fine but try to not to put too many things together as it becomes much harder to check the output.*

Prompt: "show me how many rows and orders (order_id) there are for each status in order_status_events. Also show me the counts for the different numbers of events. I.e. show many how many orders have just two events; how many have exactly three events and so on".

I expect to see a table showing the number of rows and orders for each status; and then a table showing me different numbers of events and how many orders have exactly this many.

In [8]:
print("Statuses, and how many events each order has:")
display(q("SELECT status, COUNT(*) AS n_rows, COUNT(DISTINCT order_id) AS n_orders FROM order_status_events GROUP BY 1 ORDER BY 2 DESC"))
display(q("SELECT n_events, COUNT(*) AS n_orders FROM (SELECT order_id, COUNT(*) AS n_events FROM order_status_events GROUP BY 1) GROUP BY 1 ORDER BY 1"))

Statuses, and how many events each order has:


,status,n_rows,n_orders
0,placed,14494,14494
1,accepted,14154,14055
2,picked_up,13745,13624
3,delivered,13549,13409
4,cancelled,870,870


,n_events,n_orders
0,2,439
1,3,643
2,4,13058
3,5,351
4,6,3


OK looks as expected.

*Note it is two jobs to check this so I will do them seperately.*

First table shows five statuses. Mostly the number of rows that have the status is similar or the same as the number of orders that have them. For accepted, picked_up and delivered there are more rows than orders. That must mean some of these orders were done twice - i.e. an order has a delviered event twice. Maybe this is was because something was missing or the customer added something else to the order.

Second table shows that most orders had 4 events - probably placed, accepted, picked_up and delivered based on the table above it. Logically the ones that have 2 or 3 events were probably cancelled, but there are slightly more of these (643 + 439) than there are cancelled orders (870). Could investigate this more but will move on. The ones that have 5 or more events are probably ones where there are multiple accepted, picked_up and/or delivered statuses. Again, could look further but will leave for now.

## 2. Plan

*Before you write the main code, write your plan in your own words: what one row of your final table is, which tables you will join and in what order, how many rows you expect at each step, and how you will handle the problems you found.*

**Grain of the final table:** one row per **local week** (Monday start), **cuisine** and **city**.

**Decisions**

1. **Week.** Operations work in UK time, but the data are in UTC. I convert every `order_ts` to `Europe/London` and take the Monday of that local date. In summer an order at 23:30 UTC is 00:30 local on the next day, which can be the next week or month. Weeks are by when the order was *placed*. The first and last weeks are partial: 1 March is a Sunday, so the week starting 23 February holds that single day, and 30 June is a Tuesday.
2. **Which orders count.** Orders that were not cancelled. Cancelled orders are counted in a separate column so that nothing disappears. An order is cancelled if it has a `cancelled` event.
3. **Net revenue.** Money actually kept: all `charge` rows plus all `refund` rows (refunds are stored as negative amounts), from `payments`, after removing repeated payment rows. It includes delivery fees, because they are in the amount the customer paid. It is attributed to the week the *order* was placed, not the week the refund was paid. A cancelled order therefore nets to zero.
4. **Delivery time.** Minutes from the order being placed to its first `delivered` event, only for orders that have one. Orders with no delivered event (cancelled, or stuck) are not in the average. Times that are negative are impossible and excluded from the average, and counted.
5. **Cuisine and city.** The restaurant's cuisine and city **as at the order date**, using `effective_from`, because restaurants move. Labels are trimmed and standardised.

*As you can see there is way more work than can be done in an hour (or an hour minus however long we already spent on understanding the data). Do not worry about this it is entirely by design and you can choose what you want to spend your time doing.* **NO-ONE IS EXEPCTED TO DO EVERYTHING AND I WOULD HAVE SERIOUS QUESTIONS TO ASK IF ANYONE CAN DOES!!!** *I will run my plan as making a series of silver tables. This is not expected necessarily and we don't really know enough yet for this to be actual silver tables - and we are doing exploration not exploitation at this stage. However, it shows how the ideas relate.*

**Plan for building** (expected row counts in brackets)

- `silver_restaurants` (one row per restaurant version, clean labels)
- `silver_orders` (one row per order; local time and week; restaurant version as at order date) [= rows in `orders`]
- `silver_order_events` (one row per order: cancelled flag, first delivered time, final status) [= rows in `orders`]
- `silver_order_payments` (one row per order: gross, refunds, net) [no more rows than `orders`]
- `silver_order` joins them: one row per order [= rows in `orders`]. The row count after every join must not change.
- `gold_weekly_ops`: grouped from `silver_order`.

## 3. Build

*Build as far as you can in the time. Short, re-runnable cells are better than one long cell.*

*Again I'm going to do two things at a time or else this notebook will get very long. You can do the same, or do one at a time, just don't do 7 at a time or whatever.*

PROMPT: "please clean ip the text of the restaurants table. For city and cuisine make sure both have no extra whitespaces and they are lower case with the first letter capitalised. Please count how many there are of each cusine and city afterwards so I can check the results. Create a table called silver_restaurants from this and show me the first five rows."

Expect to see a table showing cleaned up cusines first of all and their counts, and then a table of cities and their counts. Also the first five rows of the silver_restaurants table.

In [9]:
# --- silver_restaurants: clean labels, keep every version with its start date ---
r = tables["restaurants"].copy()
r["cuisine"] = r["cuisine"].str.strip().str.title()
r["city"] = r["city"].str.strip().str.title()
display(r.head())

con.register("silver_restaurants", r)
display(q("SELECT cuisine, COUNT(*) AS n FROM silver_restaurants GROUP BY 1 ORDER BY 1"))
display(q("SELECT city, COUNT(*) AS n FROM silver_restaurants GROUP BY 1 ORDER BY 1"))

,restaurant_id,restaurant_name,cuisine,city,effective_from
0,R064,Fresh Vegan 64,Vegan,London,2025-01-01
1,R012,Royal Burgers 12,Burgers,Bristol,2025-01-01
2,R028,Grand Burgers 28,Burgers,Birmingham,2025-01-01
3,R007,Royal Japanese 7,Japanese,Bristol,2025-01-01
4,R026,Corner Indian 26,Indian,London,2025-01-01


,cuisine,n
0,Burgers,8
1,Chinese,8
2,Indian,8
3,Italian,8
4,Japanese,9
5,Mexican,8
6,Thai,9
7,Vegan,9


,city,n
0,Birmingham,12
1,Bristol,7
2,Leeds,6
3,London,24
4,Manchester,18


Now lets make sure the date (effective_from) is stored as a date.

Prompt: "change the effective_from value to a datetime object and print the datatype before and after".

Expect to see a print of a string data type (object in pandas) and then a pd.datetime data type:

In [10]:
print(r["effective_from"].dtype)
r["effective_from"] = pd.to_datetime(r["effective_from"])
print(r["effective_from"].dtype)

object
datetime64[ns]


Cool that all worked. Now let's change the date object to a date as well.

*I'll also make them specifically UK time zone as they say "UTC" - universal time. Not that I would expect you to necessarily but its a thing that should probably be done*.

We also need to get the start of the week as the Monday before the date.

Prompt: "from the orders table, please convert orders_ts_utc to a datetime object in UK timezone as a new column. create a field called week_start that shows the start of the week which should be the Monday before the date or the date if it is a Monday. print the top 5 rows so I can check it worked. Then save this as silver_orders a new table."

In [11]:
# --- silver_orders: parse UTC, convert to UK local time, derive local date and week ---
o = tables["orders"].copy()
o["order_ts_utc"] = pd.to_datetime(o["order_ts"], format="%Y-%m-%dT%H:%M:%SZ")
local = o["order_ts_utc"].dt.tz_localize("UTC").dt.tz_convert("Europe/London").dt.tz_localize(None)
o["order_ts_local"] = local
o["order_date_local"] = local.dt.normalize()
# Calculate the Monday of the week
o["week_start"] = o["order_date_local"] - pd.to_timedelta(o["order_date_local"].dt.weekday, unit='D')

display(o.head())

con.register("silver_orders", o)

,order_id,customer_id,restaurant_id,order_ts,delivery_fee,order_ts_utc,order_ts_local,order_date_local,week_start
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49,2026-03-01 09:13:23,2026-03-01 09:13:23,2026-03-01,2026-02-23
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49,2026-03-01 09:15:33,2026-03-01 09:15:33,2026-03-01,2026-02-23
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00,2026-03-01 09:23:42,2026-03-01 09:23:42,2026-03-01,2026-02-23
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49,2026-03-01 09:28:14,2026-03-01 09:28:14,2026-03-01,2026-02-23
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00,2026-03-01 11:32:35,2026-03-01 11:32:35,2026-03-01,2026-02-23


Date times look good. The week_start column looks OK (didn't check a calendar but I probably should have.

Now we can deal with the order status events.

*Note there were lots of ways I could have done this and this is definitely not the simplest but it might be the "best". Other simpler fixes might have included just keeping any orders that have exactly four events or are cancelled. Or maybe keeping only those that are cancelled or delivered. I am not expecting you to have done exactly what I did here but this is a model answer so it made sense for me to give a model answer.*

Prompt: "create a table called silver_order_events from the order_status_events dataframe. First, copy the dataframe and format the event_ts column as a datetime object. Deduplicate the distinct order_id, status, and event_ts values. Rank each status stage numerically with placed as 1, accepted as 2, picked_up as 3, delivered as 4, and cancelled as 5. Group by order_id to select the final status based on the maximum stage, determine if the order was ever cancelled, and find the minimum delivered timestamp for orders that reached the delivered stage. Finally, display a count of orders grouped by final_status and is_cancelled sorted by the status stage. Print the counts of the final statuses and if they were cancelled."

*Its a fairly crazy prompt. Again you should do something simpler ... e.g. "Create a table called silver_order_events from the order_status_events dataframe. Keep only the rows where the status is 'cancelled' or 'delivered'. Group the data by order_id, setting 'is_cancelled' to true if any event was cancelled, and find the minimum event_ts as delivered_ts_utc for the delivered events."*



In [12]:
# --- silver_order_events: remove repeated rows, then one row per order ---
e = tables["order_status_events"].copy()
e["event_ts"] = pd.to_datetime(e["event_ts"], format="%Y-%m-%dT%H:%M:%SZ")
con.register("_events", e)
con.sql("""
CREATE OR REPLACE TABLE silver_order_events AS
WITH dedup AS (SELECT DISTINCT order_id, status, event_ts FROM _events),
ranked AS (
  SELECT *, CASE status WHEN 'placed' THEN 1 WHEN 'accepted' THEN 2 WHEN 'picked_up' THEN 3
                        WHEN 'delivered' THEN 4 WHEN 'cancelled' THEN 5 END AS stage
  FROM dedup)
SELECT order_id,
       arg_max(status, stage)                        AS final_status,
       bool_or(status = 'cancelled')                 AS is_cancelled,
       MIN(event_ts) FILTER (WHERE status = 'delivered') AS delivered_ts_utc
FROM ranked GROUP BY order_id
""")
display(q("SELECT final_status, is_cancelled, COUNT(*) AS n FROM silver_order_events GROUP BY ALL ORDER BY 1"))

,final_status,is_cancelled,n
0,cancelled,True,870
1,delivered,False,13409
2,picked_up,False,215


OK for payments I want to drop duplicates. Let's do this and check.

Prompt: "please drop duplicates from the payments table and check that payment_id is now unique".

Expect to see all records are now unique.

In [13]:
p = tables["payments"].drop_duplicates()                      # identical rows only
if p["payment_id"].is_unique:
  print("all unique")
else:
  print ("not unique")

all unique


All good.

Now let's create a silver order payments table with what we charged the customer, a refund if given, and then the net revenue (charged less refunded). We will create a new table with this.

Prompt: please create a table called silver_order_payments aggregated by the order_id, with the sum of the amount for that order if the payment_type is charged (as gross_charged), the amount refunded if any as refunded, and the net_revenue as gross_charged plus refunded (as refunded is a negative number). The final table should have the columns order_id, gross_charged, refunded and net_revenue. Please print the first five rows of the new table so I can check."

I expect to see this table and that any refunds look right.

In [14]:
con.register("_payments", p)
con.sql("""
CREATE OR REPLACE TABLE silver_order_payments AS
SELECT order_id,
       ROUND(SUM(amount) FILTER (WHERE payment_type = 'charge'), 2)  AS gross_charged,
       ROUND(COALESCE(SUM(amount) FILTER (WHERE payment_type = 'refund'), 0), 2) AS refunded,
       ROUND(SUM(amount), 2)                                          AS net_revenue
FROM _payments GROUP BY order_id
""")

display(q("SELECT * FROM silver_order_payments LIMIT 5"))

,order_id,gross_charged,refunded,net_revenue
0,O010295,19.49,0.00,19.49
1,O004520,63.49,-10.40,53.09
2,O008390,30.49,-4.95,25.54
3,O004635,17.49,-17.49,0.00
4,O011166,70.99,-70.99,0.00


OK great everything looks good.

*Obviously in the real world we have probably gone way over the hour but let's continue.*

Now we can update the orders table. This should join orders, events and payments together.

Prompt: "create a table called silver_order. This should be a left join on silver_orders with silver_order_events and silver_order_payments. order_id is the primary key on these tables. Include the order_id, order_date_local, order_ts_utc from silver_orders, the final_status and is_cancelled from silver_order_events and the gross_charged, refunded and net_revenue from silver_order_payments. Also include the difference in time between order_ts_utc (from silver_orders) with delivered_ts_utc from events as delivery minutes. print the number of rows in silver_order and in orders so we can check they are the same".

Expect the same number of rows so that nothing broke

In [15]:
# --- silver_order: one row per order. Row count must stay equal to orders. ---
# Register the orders dataframe 'o' as 'silver_orders' so the SQL query below can resolve it
con.register("silver_orders", o)

con.sql("""
CREATE OR REPLACE TABLE silver_order AS
SELECT o.order_id, o.order_date_local, o.week_start, o.order_ts_utc,
       e.final_status, e.is_cancelled,
       date_diff('second', o.order_ts_utc, e.delivered_ts_utc) / 60.0 AS delivery_minutes,
       p.gross_charged, p.refunded, p.net_revenue,
       r.cuisine, r.city
FROM silver_orders AS o
LEFT JOIN silver_order_events   AS e USING (order_id)
LEFT JOIN silver_order_payments AS p USING (order_id)
LEFT JOIN silver_restaurants AS r USING (restaurant_id)
""")
n_orders = len(tables["orders"])
n_silver = q("SELECT COUNT(*) AS n FROM silver_order")["n"][0]

print("silver_order rows:", n_silver, "= orders rows:", n_orders)

silver_order rows: 14852 = orders rows: 14494


You might note that the row numbers do not match! I will leave that as an Easter Egg for anyone who wants to work out why and how to fix it (but its not trivial so be warned).

Last we can create the gold table.

*Insane work if you got this far without a time machine.*

Prompt: "create a table called gold_weekly_ops. This should come from silver_order and be aggregated by week_start, cuisine and city. Include the count of records where the order is not is_cancelled = True as orders; the count of records where the order is is_cancelled = True as orders_cancelled; the sum of net_revenue rounded to two decimal places as net_revenue; the average of delivery_minutes for all records greater than 0 as avg_delivery_minutes; and the count of records where the delivery_minutes are greater than 0 as n_delivery_timed. Please join Print the first 10 records to show it worked and the total number of rows."

I'm expecting to see a lovely, aggregated gold table.

In [16]:
# --- gold_weekly_ops ---
con.sql("""
CREATE OR REPLACE TABLE gold_weekly_ops AS
SELECT week_start, cuisine, city,
       COUNT(*) FILTER (WHERE NOT is_cancelled)                          AS orders,
       COUNT(*) FILTER (WHERE is_cancelled)                              AS orders_cancelled,
       ROUND(SUM(net_revenue), 2)                                         AS net_revenue,
       ROUND(AVG(delivery_minutes) FILTER (WHERE delivery_minutes > 0), 1) AS avg_delivery_minutes,
       COUNT(*) FILTER (WHERE delivery_minutes > 0)                       AS n_delivery_timed
FROM silver_order
GROUP BY week_start, cuisine, city
""")
gold = q("SELECT * FROM gold_weekly_ops")
display(gold.head(10))
print(len(gold), "rows")

,week_start,cuisine,city,orders,orders_cancelled,net_revenue,avg_delivery_minutes,n_delivery_timed
0,2026-02-23,Burgers,London,11,0,396.42,39.7,11
1,2026-02-23,Chinese,London,7,1,284.93,35.1,7
2,2026-02-23,Burgers,Birmingham,4,0,125.99,38.8,4
3,2026-02-23,Mexican,Manchester,17,0,518.32,36.6,17
4,2026-02-23,Indian,London,7,1,215.56,30.7,7
5,2026-02-23,Burgers,Leeds,0,1,0.00,NaN,0
6,2026-02-23,Thai,Manchester,4,0,175.42,40.9,3
7,2026-02-23,Chinese,Birmingham,6,1,117.46,34.0,6
8,2026-02-23,Italian,Manchester,5,0,208.95,36.5,5
9,2026-02-23,Burgers,Bristol,3,1,66.77,40.2,3


566 rows


Amazing! Well actually it is still wrong for the Easter Egg reason above but its definitely good enough.

*Let me stress again YOU ARE NOT EXPECTED TO GET THIS FAR OR TO BE AS COMPLICATED AS I HAVE GONE HERE - IT IS A MODEL ANSWER AND WOULD SCORE 100 MARKS AND PROBABLY LAUNCH AN ACADEMIC CONDUCT PANEL AS TO HOW YOU POSSIBLY FINISHED IN THIS TIME.*

## 4. Check

*Show how you know your answer is right. What did you compare, count or spot check, and what did the checks show? If a check failed, say what you changed.*

Again, I did these as I went along so nothing included here. You can include them in text like I did this time or at the end here as on the previous notebook. Either is fine just make sure you show the checks.

## 5. Prompts and AI notes

*Paste each prompt you used, as you sent it. Then note what you accepted, what you changed or rejected, and why. Include at least one example where the AI was wrong or needed correcting, if that happened.*

Again, I did these as I went along so nothing included here. You can include them in text like I did this time or at the end here as on the previous notebook. Either is fine just make sure you show the prompts.

## 6. What I could not verify

*Be honest about what remains unchecked, and what you would do next with more time.*

**Example - just list anything you saw as a problem but didn't fix. Also can be in text as you go along rather than at the end if you prefer**

- I cannot tell from the data whether the 12 deliveries over 5 hours are real (a late delivery) or errors. I kept them for now.
- I did not fully investigate why some statuses repeated, or whether ones with less than 3 statuses were all cancelled.
- I attributed refunds to the week the order was placed. A finance team may prefer the week the refund was paid, which would give different weekly figures but the same total.
- I did not check that charges equal the sum of `order_lines` plus the delivery fee. That would be a useful reconciliation with more time.
- No automated tests are written. With more time I would add tests for: unique order id in `silver_order`, no null cuisine or city, row count equals `orders`, and gold net revenue equals cleaned payments.
- I did not fix the problem with merging restaurants leading to a different number of rows than in the orders table.